# SOXL paper/sim replay production candidate v1

目标：把已经验证的 `schedule -> dashboard -> paper intent` 链路固定成 production 候选 notebook。

边界：v1 只运行 paper/sim，不连接真实 broker，不触发真实下单。生产晋升前仍需更严格模型、风控和外部调度。

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.api import (
    PaperBrokerAdapter,
    PaperMarketDataAdapter,
    build_target_position_intent_from_dashboard,
    execute_paper_intent,
    write_intent_ledger,
)
from src.core import DatasetSnapshot, publish_production_manifest
from src.data import load_soxl_support_frame
from src.schedule import (
    DataUpdatePipelineConfig,
    ModelDashboardPipelineConfig,
    run_data_update_once,
    run_model_dashboard_refresh_once,
)

CARD = Path('production/paper_sim_replay_v1')
CARD.mkdir(parents=True, exist_ok=True)
notifications = []
data_notifications = []
config = ModelDashboardPipelineConfig(
    name='production_paper_sim_replay_v1',
    card_dir=CARD / 'matrix_card',
    dashboard_output_path=CARD / 'dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'random_forest_clf'),
    top_n_factors=3,
    dashboard_top_n=5,
    notify_details=True,
    retries=1,
)
market_data_config = DataUpdatePipelineConfig(
    name='production_paper_replay_data_update_v1',
    dataset_key='daily.soxl_paper_replay',
    dataset_name='paper_replay_soxl_daily',
    universe=('SOXL',),
    output_dir=CARD / 'data' / 'snapshots' / 'paper_replay_soxl_daily' / config.end,
    registry_path=CARD / 'data' / 'data_registry.json',
    calendar_path='data/qlib/soxl/calendars/day.txt',
    source_paths=('src.data.load_soxl_support_frame',),
    start=config.start,
    end=config.end,
    as_of=config.end,
    published_at=f'{config.end}T21:00:00Z',
    notify_details=True,
    retries=1,
)
config, market_data_config


(ModelDashboardPipelineConfig(name='production_paper_sim_replay_v1', card_dir=PosixPath('production/paper_sim_replay_v1/matrix_card'), dashboard_output_path=PosixPath('production/paper_sim_replay_v1/dashboard_export.json'), start='2026-01-01', end='2026-06-09', train_end='2026-04-30', model_names=('ridge_mse', 'random_forest_clf'), top_n_factors=3, dashboard_top_n=5, attribution_card_dir=None, dashboard_page_output_path=None, dashboard_page_title='SOXL Dashboard', analysis_manifest_preflight=False, analysis_manifest_auto_backfill=False, analysis_manifest_dataset_name='schedule_dashboard_refresh', analysis_manifest_universe=('SOXL',), analysis_manifest_calendar_path='', notify_details=True, retries=1),
 DataUpdatePipelineConfig(name='production_paper_replay_data_update_v1', dataset_key='daily.soxl_paper_replay', dataset_name='paper_replay_soxl_daily', universe=('SOXL',), output_dir=PosixPath('production/paper_sim_replay_v1/data/snapshots/paper_replay_soxl_daily/2026-06-09'), registry_pa

In [2]:
schedule_result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / 'locks',
    notify=notifications.append,
)
schedule_result

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


JobResult(name='production_paper_sim_replay_v1', status='success', payload={'matrix_card': 'production/paper_sim_replay_v1/matrix_card', 'dashboard_export': 'production/paper_sim_replay_v1/dashboard_export.json', 'dashboard_schema_version': 'dashboard.model_matrix.v1', 'dashboard_artifact_id': 'matrix_card', 'dashboard_manifest_kind': 'analysis_manifest', 'dashboard_manifest_path': 'production/paper_sim_replay_v1/matrix_card/analysis_manifest.json', 'dashboard_manifest_hash': '4d16a54a928730947e024297c634659ca6ecb1f8d575073d6878666e55bf2b46', 'dashboard_manifest_producer': 'src.analysis.exports', 'dashboard_page': '', 'dashboard_page_schema_version': '', 'dashboard_view_model_schema_version': '', 'dashboard_page_model_card_count': 0, 'top_model': 'ridge_mse__ic_weighted_composite', 'attribution_card': '', 'attribution_status': 'not_provided', 'matrix_status': 'feature_set_matrix_completed', 'variant_count': 6, 'top_failure_mode': 'return_stability_lag', 'top_failure_segment': 'stress',

In [3]:
dashboard_payload = json.loads((CARD / 'dashboard_export.json').read_text(encoding='utf-8'))
intent = build_target_position_intent_from_dashboard(
    dashboard_payload,
    instrument='SOXL',
    min_position=0.0,
    max_position=0.75,
)
intent.to_dict()

{'instrument': 'SOXL',
 'target_position': 0.75,
 'source_model': 'ridge_mse__ic_weighted_composite',
 'source_artifact': 'matrix_card',
 'mode': 'paper',
 'schema_version': 'api.execution_intent.v1',
 'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
 'metadata': {'execution_gate': 'paper_review',
  'raw_avg_position': 0.7518156857658157,
  'min_position': 0.0,
  'max_position': 0.75,
  'base_model': 'ridge_mse',
  'feature_set': 'ic_weighted_composite',
  'stability_score': 29.940062203007624,
  'max_drawdown_pct': 29.020142371836933,
  'promotion_decision': 'promote_candidate',
  'promotion_score': 29.940062203007624,
  'promotion_blockers': '',
  'dashboard_schema_version': 'dashboard.model_matrix.v1',
  'run_status': 'complete',
  'run_trained': True,
  'run_predicted': True,
  'run_backtested': True,
  'run_objective_evaluated': True,
  'run_promotion_evaluated': True,
  'run_analysis_recorded': True,
  'run_manifested': True,
  'model_run_schema_version': 'model.run_ledger_

In [4]:
def _paper_replay_frame_provider(_: DataUpdatePipelineConfig) -> pd.DataFrame:
    frame = load_soxl_support_frame(start=config.start, end=config.end).copy()
    frame['instrument'] = intent.instrument
    return frame.loc[:, ['date', 'instrument', 'open', 'high', 'low', 'close', 'volume', 'soxx_close', 'soxx_volume', 'vix']]


def _paper_replay_calendar_provider(_: DataUpdatePipelineConfig) -> pd.DatetimeIndex:
    frame = load_soxl_support_frame(start=config.start, end=config.end)
    return pd.DatetimeIndex(pd.to_datetime(frame['date']), name='date')


data_update_result = run_data_update_once(
    market_data_config,
    lock_dir=CARD / 'locks',
    notify=data_notifications.append,
    frame_provider=_paper_replay_frame_provider,
    calendar_provider=_paper_replay_calendar_provider,
)
market_data = PaperMarketDataAdapter.from_registry(
    market_data_config.registry_path,
    dataset_key=market_data_config.dataset_key,
)
market_quote = market_data.latest_quote(intent.instrument)

broker = PaperBrokerAdapter(cash=10000.0, shares=0)
execution_result = execute_paper_intent(intent, broker=broker, price=market_quote.price)
(CARD / 'execution_intent.json').write_text(json.dumps(intent.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'market_quote.json').write_text(json.dumps(market_quote.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'paper_execution_result.json').write_text(json.dumps(execution_result.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
write_intent_ledger(CARD / 'intent_ledger.json', intent=intent, result=execution_result)
{
    'data_update': data_update_result.payload,
    'market_quote': market_quote.to_dict(),
    'execution_result': execution_result.to_dict(),
}


{'data_update': {'schema_version': 'schedule.data_update_result.v1',
  'dataset_key': 'daily.soxl_paper_replay',
  'dataset_name': 'paper_replay_soxl_daily',
  'status': 'published',
  'manifest_path': 'production/paper_sim_replay_v1/data/snapshots/paper_replay_soxl_daily/2026-06-09/data_update_manifest.json',
  'manifest_hash': 'c2d66c782be2de4adb47554cd4c5211481a9be3c110bc90fdc7da6f855a317a7',
  'dataset_schema_hash': '956800a74aa0ed21477e2ffba40a4359641ee6bcfb4347f15d6eae3ee118b5a3',
  'registry_path': 'production/paper_sim_replay_v1/data/data_registry.json',
  'start': '2026-01-02',
  'end': '2026-06-09',
  'as_of': '2026-06-09',
  'row_count': 109,
  'column_count': 10,
  'frame_schema_hash': 'cc88b355f6e3340e9bca290d2bc329530b03ecafc6a5f78eb9be4daee9b08f71',
  'dropped_non_calendar_rows': 0,
  'published_at': '2026-06-09T21:00:00Z',
  'attempts': 1},
 'market_quote': {'schema_version': 'api.market_quote.v1',
  'instrument': 'SOXL',
  'date': '2026-06-09',
  'price': 201.68,
  'fi

In [5]:
intent_metadata = dict(intent.metadata or {})
analysis_overview = dashboard_payload.get('analysis_overview', {})
top_gap = (analysis_overview.get('largest_relative_return_spread_segments') or [{}])[0]
summary = {
    'schema_version': 'production.paper_sim_replay.v1',
    'status': 'paper_replay_completed',
    'mode': intent.mode,
    'live_broker_enabled': False,
    'instrument': intent.instrument,
    'source_model': intent.source_model,
    'target_position': intent.target_position,
    'execution_gate': intent_metadata.get('execution_gate', ''),
    'promotion_decision': intent_metadata.get('promotion_decision', ''),
    'promotion_blockers': intent_metadata.get('promotion_blockers', ''),
    'analysis_overview_schema': intent_metadata.get('analysis_overview_schema_version', ''),
    'analysis_top_gap_segment': intent_metadata.get('analysis_top_gap_segment', ''),
    'analysis_top_gap_spread': intent_metadata.get('analysis_top_gap_spread', 0.0),
    'analysis_top_gap_best_model': intent_metadata.get('analysis_top_gap_best_model', ''),
    'analysis_top_gap_worst_model': intent_metadata.get('analysis_top_gap_worst_model', ''),
    'analysis_top_weakness_model': intent_metadata.get('analysis_top_weakness_model', ''),
    'analysis_top_weakness_decision_hint': intent_metadata.get('analysis_top_weakness_decision_hint', ''),
    'backtest_schema_version': intent_metadata.get('backtest_schema_version', ''),
    'backtest_walk_forward_fold_count': intent_metadata.get('backtest_walk_forward_fold_count', 0),
    'backtest_slippage_bps': intent_metadata.get('backtest_slippage_bps', 0.0),
    'market_data_adapter': market_data.adapter_name,
    'market_data_dataset_key': market_data_config.dataset_key,
    'market_data_registry_path': str(market_data_config.registry_path),
    'market_data_manifest_path': data_update_result.payload.get('manifest_path', ''),
    'market_data_manifest_hash': data_update_result.payload.get('manifest_hash', ''),
    'market_quote': market_quote.to_dict(),
    'paper_shares': execution_result.snapshot.shares,
    'paper_position': execution_result.snapshot.position,
    'schedule_status': schedule_result.status,
    'schedule_attempts': schedule_result.payload.get('attempts'),
    'data_update_status': data_update_result.status,
    'data_update_attempts': data_update_result.payload.get('attempts'),
    'data_update_row_count': data_update_result.payload.get('row_count'),
    'data_update_as_of': data_update_result.payload.get('as_of'),
    'notifications': notifications,
    'data_notifications': data_notifications,
    'dashboard_export': str(CARD / 'dashboard_export.json'),
    'intent_ledger': str(CARD / 'intent_ledger.json'),
    'matrix_card': str(CARD / 'matrix_card'),
    'guardrail': intent.guardrail,
    'analysis_overview': analysis_overview,
    'top_gap': top_gap,
}
(CARD / 'production_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

production_dataset = DatasetSnapshot(
    name=market_data_config.dataset_name,
    universe=market_data_config.universe,
    start=data_update_result.payload.get('start', config.start),
    end=data_update_result.payload.get('end', config.end),
    calendar_path=market_data_config.calendar_path,
    data_paths=(str(market_data_config.registry_path), data_update_result.payload.get('manifest_path', '')),
    metadata={
        'market_data_dataset_key': market_data_config.dataset_key,
        'market_data_manifest_hash': data_update_result.payload.get('manifest_hash', ''),
        'dashboard_manifest_hash': schedule_result.payload.get('dashboard_manifest_hash', ''),
        'dashboard_manifest_kind': schedule_result.payload.get('dashboard_manifest_kind', ''),
    },
)
production_manifest = publish_production_manifest(
    strategy_id='paper_sim_replay_v1',
    output_dir=CARD,
    registry_path=Path('production') / 'production_registry.json',
    notebook_path='paper_sim_replay_v1.ipynb',
    dataset=production_dataset,
    paths={
        'summary': 'production_summary.json',
        'dashboard_export': 'dashboard_export.json',
        'intent_ledger': 'intent_ledger.json',
        'execution_intent': 'execution_intent.json',
        'market_quote': 'market_quote.json',
        'paper_execution_result': 'paper_execution_result.json',
        'data_registry': 'data/data_registry.json',
        'data_update_manifest': str(Path(data_update_result.payload.get('manifest_path', '')).relative_to(CARD)),
        'analysis_manifest': 'matrix_card/analysis_manifest.json',
        'model_artifact_manifest': 'matrix_card/artifact_manifest.json',
    },
    metadata={
        'status': summary['status'],
        'mode': summary['mode'],
        'live_broker_enabled': summary['live_broker_enabled'],
        'instrument': summary['instrument'],
        'source_model': summary['source_model'],
        'target_position': summary['target_position'],
        'execution_gate': summary['execution_gate'],
        'promotion_decision': summary['promotion_decision'],
        'market_data_dataset_key': summary['market_data_dataset_key'],
        'market_quote_date': summary['market_quote']['date'],
        'market_quote_price': summary['market_quote']['price'],
        'paper_position': summary['paper_position'],
        'paper_shares': summary['paper_shares'],
        'schedule_status': summary['schedule_status'],
        'data_update_status': summary['data_update_status'],
        'analysis_overview_schema': summary['analysis_overview_schema'],
        'backtest_schema_version': summary['backtest_schema_version'],
    },
    published_at=f'{config.end}T21:30:00Z',
)
summary['production_manifest_path'] = str(CARD / 'production_manifest.json')
summary['production_manifest_hash'] = production_manifest['manifest_hash']
summary['production_registry_path'] = 'production/production_registry.json'
(CARD / 'production_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'production.paper_sim_replay.v1',
 'status': 'paper_replay_completed',
 'mode': 'paper',
 'live_broker_enabled': False,
 'instrument': 'SOXL',
 'source_model': 'ridge_mse__ic_weighted_composite',
 'target_position': 0.75,
 'execution_gate': 'paper_review',
 'promotion_decision': 'promote_candidate',
 'promotion_blockers': '',
 'analysis_overview_schema': 'analysis.overview.v1',
 'analysis_top_gap_segment': 'downtrend',
 'analysis_top_gap_spread': 47.970097543991834,
 'analysis_top_gap_best_model': 'random_forest_clf__ic_weighted_composite',
 'analysis_top_gap_worst_model': 'ridge_mse__all_features',
 'analysis_top_weakness_model': 'ridge_mse__all_features',
 'analysis_top_weakness_decision_hint': 'block_or_rework',
 'backtest_schema_version': 'backtest.config.v1',
 'backtest_walk_forward_fold_count': 5,
 'backtest_slippage_bps': 1.0,
 'market_data_adapter': 'paper_market_data',
 'market_data_dataset_key': 'daily.soxl_paper_replay',
 'market_data_registry_path': 'prod

In [6]:
assert summary['execution_gate'] in {'paper_review', 'blocked_to_min_position'}
assert summary['analysis_overview_schema'] == 'analysis.overview.v1'
assert summary['promotion_decision']
assert summary['backtest_schema_version'] == 'backtest.config.v1'
assert summary['backtest_walk_forward_fold_count'] >= 1
assert summary['market_data_dataset_key'] == 'daily.soxl_paper_replay'
assert summary['data_update_status'] == 'success'
assert summary['production_manifest_hash'] == production_manifest['manifest_hash']
assert production_manifest['schema_version'] == 'production.manifest.v1'
assert production_manifest['paths']['intent_ledger'] == 'intent_ledger.json'
assert production_manifest['paths']['data_registry'] == 'data/data_registry.json'
assert summary['market_quote']['source_path'].endswith('canonical_frame.csv')
assert 'data/snapshots/' in summary['market_quote']['source_path']
assert 'regime_gap=' in ' '.join(summary['notifications'])
assert 'dataset=daily.soxl_paper_replay' in ' '.join(summary['data_notifications'])
assert intent_metadata['execution_gate'] == summary['execution_gate']

{
    'dashboard_top_models': dashboard_payload['top_models'][:3],
    'feature_set_summary': dashboard_payload['feature_set_summary'],
    'execution_gate': summary['execution_gate'],
    'promotion_decision': summary['promotion_decision'],
    'analysis_top_gap_segment': summary['analysis_top_gap_segment'],
    'backtest_schema_version': summary['backtest_schema_version'],
    'market_data_dataset_key': summary['market_data_dataset_key'],
    'market_quote_source_path': summary['market_quote']['source_path'],
    'production_manifest_hash': summary['production_manifest_hash'],
    'ledger_exists': (CARD / 'intent_ledger.json').exists(),
    'live_broker_enabled': summary['live_broker_enabled'],
}


{'dashboard_top_models': [{'rank': 1,
   'model': 'ridge_mse__ic_weighted_composite',
   'base_model': 'ridge_mse',
   'feature_set': 'ic_weighted_composite',
   'return_pct': 38.31176843102375,
   'benchmark_return_pct': 58.118381163943475,
   'relative_return_pct': -19.806612732919724,
   'max_drawdown_pct': 29.020142371836933,
   'sharpe': 3.115575854413103,
   'avg_position': 0.7518156857658157,
   'turnover': 1.6261434036754974,
   'stability_score': 29.940062203007624},
  {'rank': 2,
   'model': 'ridge_mse__top_ic_factors',
   'base_model': 'ridge_mse',
   'feature_set': 'top_ic_factors',
   'return_pct': 37.09597053157543,
   'benchmark_return_pct': 58.118381163943475,
   'relative_return_pct': -21.022410632368043,
   'max_drawdown_pct': 29.12731437649879,
   'sharpe': 3.0519794135557325,
   'avg_position': 0.7473726988299527,
   'turnover': 1.949158985539,
   'stability_score': 27.308303725431273},
  {'rank': 3,
   'model': 'random_forest_clf__ic_weighted_composite',
   'base_m

## 中文结论

- 做了什么：运行 production 候选链路，完成 model/dashboard schedule 刷新、data update schedule 发布 canonical snapshot 与 data registry、paper intent 生成、registry-backed paper market-data quote、PaperBrokerAdapter 模拟执行，并发布 `production_manifest.json` 与 `production_registry.json`。
- 核心指标：source model、target_position、market quote、paper shares、paper position、execution_gate、promotion_decision、backtest schema、market_data_dataset_key、data_update_status、production_manifest_hash 均写入 `production_summary.json` 与相关 sidecar。
- regime 差异：dashboard export 保留 `analysis_overview.v1`，paper intent metadata 记录最大 `regime_gap`、top weakness 和 promotion 决策，生产回放不再只看全区间仓位。
- 是否晋升：否。本 notebook 是 paper/sim production candidate，不是 live 生产策略。
- 风险：`live_broker_enabled=False`；guardrail=`v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。`；quote 来自 as-of data registry current snapshot，生产 manifest 仅登记 paper/sim 证据，仍需外部调度、审计留痕和 live adapter 权限隔离后才能讨论真实 broker。
- 下一步：补外部调度入口，并把 production manifest current 指针接入 dashboard/schedule 的运行健康检查。
